# MP9 — BTS Analysis


In [ ]:
import torch

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

from pathlib import Path
import json
import sys
import os
import platform
import numpy as np
import pandas as pd

from sklearn.metrics import (
    cohen_kappa_score,
    confusion_matrix,
    accuracy_score,
    f1_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


ROOT = Path.cwd().resolve()

# VS Code may launch Jupyter from the workspace root.
if not (ROOT / "gold_adjudicated.jsonl").exists():
    matches = [
        p.parent
        for p in ROOT.rglob("gold_adjudicated.jsonl")
        if ".venv" not in p.parts
    ]

    if len(matches) == 1:
        ROOT = matches[0]
    else:
        raise FileNotFoundError(
            "Cannot locate MP9_GenesisJ. "
            f"Found candidates: {matches[:10]}"
        )

PROJECT_ROOT = ROOT.parent


MP6_DIR = ROOT / "MP6_Genesis"
MP7_DIR = ROOT / "MP7_Genesis"

print("MP9 directory:", ROOT)
print("MP6 exists:", MP6_DIR.is_dir())
print("MP7 exists:", MP7_DIR.is_dir())

if not MP6_DIR.is_dir():
    raise FileNotFoundError(
        f"MP6 folder not found: {MP6_DIR}"
    )

if not MP7_DIR.is_dir():
    raise FileNotFoundError(
        f"MP7 folder not found: {MP7_DIR}"
    )


GOLD_RAW = ROOT / "gold_raw.csv"
GOLD_ADJUDICATED = ROOT / "gold_adjudicated.jsonl"
RELEASE_DATES = ROOT / "bts_release_dates.csv"
ADJUDICATION = ROOT / "adjudication.csv"
PARTNER_LABELS = ROOT / "partner_label_test.csv"
BTS_REFERENCE = ROOT / "bts_gold_reference.csv"
OVERLAP_IDS = ROOT / "overlap_ids.txt"

OUT = ROOT / "results.json"


SEED = 42
rng = np.random.default_rng(SEED)

results = {
    "status": "partial",
    "seed": SEED,
    "environment": {
        "python": platform.python_version()
    },
    "pending": []
}


def find(name):
    # Check MP9 first.
    direct = ROOT / name
    if direct.is_file():
        return direct

    # Search the actual MP6 and MP7 folders.
    matches = []

    for folder in [MP6_DIR, MP7_DIR]:
        matches.extend(
            p for p in folder.rglob(name)
            if p.is_file()
            and not any(
                excluded in p.parts
                for excluded in (
                    ".venv",
                    "node_modules",
                    ".git"
                )
            )
        )

    if not matches:
        return None

    return sorted(
        matches,
        key=lambda p: (len(p.parts), str(p))
    )[0]



def read_jsonl(path):
    path = Path(path)

    return [
        json.loads(line)
        for line in path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]



def bootstrap_ci(items, fn, B=2000, seed=SEED):
    r = np.random.default_rng(seed)
    vals = []
    items = np.asarray(items)

    for _ in range(B):
        idx = r.integers(
            0,
            len(items),
            len(items)
        )

        try:
            v = float(fn(items[idx]))

            if np.isfinite(v):
                vals.append(v)

        except (ValueError, ZeroDivisionError):
            pass

    if not vals:
        return [None, None]

    return [
        float(x)
        for x in np.quantile(
            vals,
            [0.025, 0.975]
        )
    ]



files = {
    "Gold raw": GOLD_RAW,
    "Gold adjudicated": GOLD_ADJUDICATED,
    "BTS release dates": RELEASE_DATES,
    "Adjudication": ADJUDICATION,
    "Partner labels": PARTNER_LABELS,
    "BTS reference": BTS_REFERENCE,
    "Overlap IDs": OVERLAP_IDS
}

print("\nMP9 INPUT FILES")

for name, path in files.items():
    print(
        f"{name}:",
        "FOUND" if path.is_file() else "MISSING"
    )


os.chdir(ROOT)

print("\nWorking directory:", Path.cwd())
print("Configuration complete.")


In [ ]:
from pathlib import Path
import sys
import json

ROOT = Path("/Users/genesisjoseph/Downloads/CIS731/cis531-731-2026_fall/MP9_GenesisJ")
MP7_DIR = ROOT / "MP7_Genesis"

sys.path.insert(0, str(MP7_DIR))

from classifiers import Embedder, KNNClassifier

# Load existing MP7 exemplars
exemplar_path = MP7_DIR / "data" / "exemplars.jsonl"

with open(exemplar_path, encoding="utf-8") as f:
    exemplars = [json.loads(line) for line in f if line.strip()]

print("Loaded exemplars:", len(exemplars))

# Load embedding model
embedder = Embedder("sentence-transformers/all-MiniLM-L6-v2")

# Build FAISS index
faiss_classifier = KNNClassifier(embedder, n_neighbors=5)
faiss_classifier.fit(exemplars)

print("FAISS index built successfully")
print("Indexed vectors:", faiss_classifier.index.ntotal)


## Gold annotations


In [ ]:
raw_path=find('gold_raw.csv')
gold_path=find('gold_adjudicated.jsonl') or find('gold_adjudicated(1).jsonl')
partner_path=find('partner_label_test.csv')
adj_path=find('adjudication.csv')
assert all([raw_path,gold_path,partner_path,adj_path]), 'Required MP9 files missing: gold_raw.csv, gold_adjudicated.jsonl, partner_label_test.csv, adjudication.csv'
raw=pd.read_csv(raw_path); gold=pd.DataFrame(read_jsonl(gold_path)); partner=pd.read_csv(partner_path); adj=pd.read_csv(adj_path)
assert len(raw)<=50 and raw.item_id.is_unique, 'CRITICAL: gold_raw must have <=50 unique rows'
assert len(gold)==len(raw) and gold.item_id.is_unique
assert set(raw.item_id)==set(gold.item_id)
assert len(partner)==20 and partner.item_id.is_unique
assert set(partner.item_id).issubset(set(raw.item_id))
assert (raw.set_index('item_id').loc[gold.item_id,'label'].to_numpy()==gold.label.to_numpy()).all(), 'Review gold/adjudication mismatch'
for _,a in adj.iterrows():
    assert gold.set_index('item_id').loc[a.item_id,'label']==a['final']
results['gold_gate']={'rows':len(raw),'passed':True,'gold_unique':True,'overlap':len(partner)}
# Preserve original timestamps as uploaded; correct date manually only when factually justified.
raw.to_csv(ROOT/'gold_raw.csv',index=False)
with open(ROOT/'gold_adjudicated.jsonl','w') as f:
    for r in gold.to_dict('records'):f.write(json.dumps(r,ensure_ascii=False)+'\n')
(ROOT/'overlap_ids.txt').write_text('\n'.join(partner.item_id)+'\n')
print('Gold gate PASS:',len(raw),'rows. Exported gold_raw.csv, gold_adjudicated.jsonl, overlap_ids.txt')


## Inter-annotator agreement


In [ ]:
overlap=raw[['item_id','label']].merge(partner,on='item_id',suffixes=('_mine','_partner'))
assert len(overlap)==20
labels=sorted(set(overlap.label_mine)|set(overlap.label_partner))
y1=overlap.label_mine.to_numpy(); y2=overlap.label_partner.to_numpy()
kappa=float(cohen_kappa_score(y1,y2,labels=labels)); agreement=float(np.mean(y1==y2))
indices=np.arange(len(overlap)); k_ci=bootstrap_ci(indices,lambda idx:cohen_kappa_score(y1[idx],y2[idx],labels=labels),B=10000)
cm=confusion_matrix(y1,y2,labels=labels)
results['iaa']={'n_overlap':20,'agreement':agreement,'kappa':kappa,'kappa_ci95':k_ci,'labels':labels,'confusion_matrix':cm.tolist(),'disagreements':overlap.loc[overlap.label_mine!=overlap.label_partner,'item_id'].tolist()}
print('Agreement:',agreement,'Kappa:',round(kappa,4),'95% CI:',k_ci)
display(pd.DataFrame(cm,index=labels,columns=labels))


## Label mapping


In [ ]:
MP7_TO_GOLD={'youth':'youth','friendship':'mutual comfort','romance':'love','self_love':'loving oneself','longing':'longing','social':'generational inequality','celebration':'joy','fame':None}
print(MP7_TO_GOLD)


## MP7 exemplars


In [ ]:
exemplar_path = find('exemplars.jsonl')
if exemplar_path is None:
    exemplars=[]
    results['pending'].append('MP7 data/exemplars.jsonl not found in workspace; provide existing MP7 folder')
    print('PENDING: no MP7 exemplars.jsonl found under', PROJECT_ROOT)
else:
    exemplars=read_jsonl(exemplar_path)
    assert exemplars and all('text' in x and 'label' in x for x in exemplars)
    print('MP7 exemplars:', len(exemplars), 'from', exemplar_path)
faiss_candidates=[p for p in PROJECT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in ('.faiss','.index') and '.venv' not in p.parts]
results['mp7']={'exemplars':len(exemplars),'exemplar_path':str(exemplar_path) if exemplar_path else None,'saved_faiss_index_candidates':[str(p) for p in faiss_candidates]}
if not faiss_candidates:
    results['pending'].append('No saved MP6/MP7 FAISS index found; retrieval below is TF-IDF proxy, not rubric-required FAISS')


## Unlabeled text


In [ ]:
pool_path=find('unlabeled_chunks.csv')
silver=None
if pool_path is None or not exemplars:
    results['pending'].append('Provide >=1000 genuine unlabeled chunks AND existing MP7 exemplars/classifier; do not substitute fabricated data')
    print('PENDING: unlabeled_chunks.csv is absent; no silver metrics fabricated.')
else:
    pool=pd.read_csv(pool_path).dropna(subset=['item_id','text'])
    assert pool.item_id.is_unique and len(pool)>=1000, 'Requires >=1000 unique real chunks'
    texts=[r['text'] for r in exemplars]
    vec=TfidfVectorizer(ngram_range=(1,2),min_df=1,stop_words='english')
    mat=vec.fit_transform(texts)
    sims=cosine_similarity(vec.transform(pool.text.astype(str)),mat)
    top=sims.argmax(axis=1)
    proto=np.array([exemplars[i]['label'] for i in top])
    # Independent keyword LF: abstain when no unique cue exists
    cues={'youth':['school','teen','young'],'friendship':['friend','together'],'romance':['love','romance'],'self_love':['myself','identity'],'longing':['miss','goodbye'],'social':['society','inequality'],'celebration':['dance','party'],'fame':['fame','success']}
    def keyword_lf(t):
        t=str(t).lower(); found=[k for k,words in cues.items() if any(w in t for w in words)]
        return found[0] if len(found)==1 else None
    lf=[keyword_lf(t) for t in pool.text]
    # Weights are declared heuristics, not optimized on gold.
    preds=[a if b is None or a==b else a for a,b in zip(proto,lf)]
    silver=pool.copy();silver['label_mp7']=preds;silver['label']=silver.label_mp7.map(MP7_TO_GOLD);silver['source']='silver';silver['method']='tfidf_exemplar_plus_keyword_weighted_vote_2_to_1'
    silver.to_csv(ROOT/'silver_labels.csv',index=False)
    results['silver']={'n':len(silver),'source':'silver','method':'TFIDF fallback, not MP7 classifier','keyword_lf_coverage':float(np.mean(pd.notna(lf))),'unmapped_count':int(silver.label.isna().sum())}
    print('Wrote silver_labels.csv:',len(silver),'rows. NOTE: replace fallback with MP7 classifier for full compliance.')


## Gold text


In [ ]:
gold_text_path=find('gold_text.csv')
gold_text=None
if gold_text_path is None:
    results['pending'].append('Provide verified gold_text.csv with item_id,text,cohort to score text predictions against gold')
    print('PENDING: no gold_text.csv; cannot compute honest silver-vs-gold agreement.')
else:
    gold_text=pd.read_csv(gold_text_path)
    assert {'item_id','text'}.issubset(gold_text.columns)
    gold_text=gold_text.merge(gold,on='item_id',validate='one_to_one')
    print('Verified gold text rows:',len(gold_text))
    if silver is not None:
        paired=gold_text[['item_id','label']].merge(silver[['item_id','label']],on='item_id',suffixes=('_gold','_silver'))
        paired=paired.dropna()
        if len(paired):
            results['silver_gold_agreement']={'n':len(paired),'accuracy':float(accuracy_score(paired.label_gold,paired.label_silver))}
        else:
            results['pending'].append('Silver and gold IDs do not overlap; generate silver predictions on gold texts separately')


## Retrieval evaluation


In [ ]:
if gold_text is None or not exemplars:
    results['pending'].append('Retrieval P@k/Recall@k and baseline comparison require gold_text.csv and relevance judgments')
    print('PENDING: gold_text.csv and MP7 exemplars required.')
else:
    train=[r for r in exemplars if MP7_TO_GOLD.get(r['label']) is not None]
    train_labels=np.array([MP7_TO_GOLD[r['label']] for r in train]); query_labels=gold_text.label.to_numpy()
    tf=TfidfVectorizer(ngram_range=(1,2),stop_words='english'); train_mat=tf.fit_transform([r['text'] for r in train]); query_mat=tf.transform(gold_text.text.astype(str))
    sim=cosine_similarity(query_mat,train_mat)
    rank=np.argsort(-sim,axis=1)
    retrieval={}
    for k in (3,5,10):
        hits=np.array([np.sum(train_labels[rank[i,:k]]==y) for i,y in enumerate(query_labels)])
        relevant=np.array([(train_labels==y).sum() for y in query_labels]); pk=hits/k; rk=hits/np.maximum(relevant,1)
        retrieval[str(k)]={'precision_at_k':float(pk.mean()),'recall_at_k':float(rk.mean()),'p_ci95':bootstrap_ci(np.arange(len(pk)),lambda ids:pk[ids].mean()),'r_ci95':bootstrap_ci(np.arange(len(rk)),lambda ids:rk[ids].mean())}
    results['retrieval_proxy_same_label']={'metrics':retrieval,'relevance':'same mapped label proxy; not independently judged PS6 relevance','index':'rebuilt TFIDF cosine; not original MP7 FAISS'}
    results['pending'].append('Run original MP7 Classifier and fixed-demo baseline on identical gold texts for fair classification comparison')
    print(json.dumps(retrieval,indent=2))


## Cohort analysis


In [ ]:
dates_path = find('bts_release_dates.csv')
assert dates_path is not None, 'Put bts_release_dates.csv alongside this notebook.'
dates = pd.read_csv(dates_path)
assert {'item_id','song_title','release_date'} <= set(dates.columns)
dates['release_date'] = pd.to_datetime(dates['release_date'], errors='raise')
assert dates.item_id.is_unique and dates.release_date.notna().all()
cohort_df = gold[['item_id','label']].merge(dates, on='item_id', validate='one_to_one', indicator=True)
assert len(cohort_df) == len(gold) == 50 and (cohort_df['_merge']=='both').all()
cohort_df = cohort_df.drop(columns='_merge')
cohort_df['release_year'] = cohort_df.release_date.dt.year
labels = sorted(cohort_df.label.unique())
counts = pd.crosstab(cohort_df.release_year, cohort_df.label).reindex(columns=labels,fill_value=0)
cohort_table = counts.div(counts.sum(axis=1), axis=0)
years = cohort_table.index.to_numpy()
print('Songs per year:', counts.sum(axis=1).to_dict())
print('Observed years:', years.tolist())
# Fixed kernels operate on adjacent observed yearly bins, requiring consecutive years.
assert np.all(np.diff(years)==1), 'Missing release year(s): do not convolve nonadjacent years as consecutive.'
V = cohort_table.to_numpy(dtype=float)
smooth = 0.25*V[:-2] + 0.5*V[1:-1] + 0.25*V[2:]
# First differences of smoothed vectors: next minus previous smoothed cohort.
drift = np.diff(smooth, axis=0)
rng = np.random.default_rng(42)
B=1000
by_year = [cohort_df.loc[cohort_df.release_year==year,'label'].to_numpy() for year in years]
boots=[]
for _ in range(B):
    vb=[]
    for arr in by_year:
        sample=rng.choice(arr,size=len(arr),replace=True)
        vb.append([(sample==lab).mean() for lab in labels])
    vb=np.asarray(vb)
    sb=.25*vb[:-2]+.5*vb[1:-1]+.25*vb[2:]
    boots.append(np.diff(sb,axis=0))
boots=np.asarray(boots)
ci_low,ci_high=np.quantile(boots,[.025,.975],axis=0)
drift_rows=[]
for i in range(len(drift)):
    for j,lab in enumerate(labels):
        drift_rows.append({'from_year':int(years[i+1]),'to_year':int(years[i+2]),'label':lab,'drift':float(drift[i,j]),'ci_lower':float(ci_low[i,j]),'ci_upper':float(ci_high[i,j])})
pd.DataFrame(drift_rows).to_csv('bts_cohort_drift.csv',index=False)
results['convolution']={'axis':'BTS original release year (project-specific VDB adaptation)', 'sample_size':len(cohort_df),'years':years.tolist(),'labels':labels,'counts':counts.to_dict(orient='index'),'smoothing_kernel':[.25,.5,.25],'difference_kernel':[-1,1],'bootstrap_replicates':B,'drift':drift_rows,'limitations':['Nonrandom 50-song gold sample','Sparse year cohorts','Yearly axis differs from prescribed release-decade axis']}
print('Saved bts_cohort_drift.csv; drift rows:',len(drift_rows))


## Forecasting


In [ ]:
try:
    import torch
    from torch import nn
except ImportError:
    results['pending'].append('Install PyTorch 2.x (CPU) and rerun learned convolution cell.')
else:
    torch.manual_seed(42)
    torch.set_num_threads(1)
    X = V.astype(np.float32)
    if len(X) < 7:
        results['pending'].append('At least 7 consecutive yearly cohorts required for exploratory learned-kernel train/test split.')
    else:
        # shape: samples, label channels, 3 prior years
        windows=np.stack([X[i:i+3].T for i in range(len(X)-3)])
        targets=X[3:]
        split=max(1,int(len(windows)*.7))
        assert split < len(windows)
        xtr=torch.tensor(windows[:split]); ytr=torch.tensor(targets[:split]); xte=torch.tensor(windows[split:]); yte=targets[split:]
        class ForecastConv(nn.Module):
            def __init__(self, channels):
                super().__init__()
                self.conv=nn.Conv1d(channels,channels,kernel_size=3,groups=channels,bias=False)
            def forward(self,x):
                return self.conv(x).squeeze(-1)
        model=ForecastConv(X.shape[1]); optimizer=torch.optim.Adam(model.parameters(),lr=.01)
        for epoch in range(300):
            optimizer.zero_grad(); loss=((model(xtr)-ytr)**2).mean(); loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad(): learned=model(xte).numpy()
        tw=windows[split:]
        # Forecast baseline using 3 observed years; weights in chronological order.
        smoothed=(tw*np.array([.25,.5,.25],dtype=np.float32)).sum(axis=2)
        # First difference [-1,1] over last two observed years; extrapolate from last value.
        difference=tw[:,:,-1]+(tw[:,:,-1]-tw[:,:,-2])
        mse=lambda pred:float(np.mean((pred-yte)**2))
        forecast={'train_windows':int(split),'test_windows':int(len(windows)-split),'test_target_years':years[3+split:].tolist(),'fixed_smoothing_forecast_mse':mse(smoothed),'first_difference_extrapolation_mse':mse(difference),'learned_conv_mse':mse(learned),'learned_weights_per_label':dict(zip(labels,model.conv.weight.detach().numpy()[:,0,:].tolist())),'limitations':['Tiny chronological test set; exploratory only','Unconstrained Conv1d outputs are not guaranteed to be probabilities']}
        results['learned_convolution']=forecast
        pd.DataFrame([{'model':k,'held_out_mse':forecast[k]} for k in ('fixed_smoothing_forecast_mse','first_difference_extrapolation_mse','learned_conv_mse')]).to_csv('bts_forecast_comparison.csv',index=False)
        print('Held-out forecast MSE:',{k:v for k,v in forecast.items() if k.endswith('_mse')})


## Results


In [ ]:
results['status']='complete' if not results['pending'] else 'partial'
results['pending']=list(dict.fromkeys(results['pending']))
OUT.write_text(json.dumps(results,indent=2,ensure_ascii=False))
print('Wrote:',OUT.resolve(),'STATUS:',results['status'])
print('PENDING ITEMS:')
for x in results['pending']:print(' -',x)
print('Instructor validator: run validate_annotation_submission locally when available; not bundled here.')


In [ ]:
import csv
import json
import sys
from pathlib import Path

import torch

torch.set_num_threads(1)

ROOT = Path(
    "/Users/genesisjoseph/Downloads/CIS731/"
    "cis531-731-2026_fall/MP9_GenesisJ"
)

sys.path.insert(0, str(ROOT / "MP7_Genesis"))

from classifiers import Embedder, KNNClassifier

# Load the original MP7 exemplars.
exemplars_path = ROOT / "MP7_Genesis/data/exemplars.jsonl"

with open(exemplars_path, encoding="utf-8") as f:
    exemplars = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

# Build the FAISS-backed classifier.
embedder = Embedder("sentence-transformers/all-MiniLM-L6-v2")

classifier = KNNClassifier(
    embedder,
    n_neighbors=5
).fit(exemplars)

# Load all 50 BTS descriptions.
with open(ROOT / "gold_text.csv", encoding="utf-8") as f:
    songs = list(csv.DictReader(f))

assert len(songs) == 50
assert all(row["text"].strip() for row in songs)

texts = [row["text"] for row in songs]

# Predict the theme of each description.
predictions = classifier.predict(texts)

# Save results without changing human labels.
output = ROOT / "mp9_faiss_predictions.csv"

with open(output, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "item_id",
        "song_title",
        "predicted_label"
    ])

    for song, prediction in zip(songs, predictions):
        writer.writerow([
            song["item_id"],
            song["song_title"],
            prediction
        ])

print("Songs classified:", len(predictions))
print("Predictions saved:", output)

for song, prediction in list(zip(songs, predictions))[:10]:
    print(song["song_title"], "->", prediction)


In [ ]:
import json
from collections import Counter

gold_path = ROOT / "gold_adjudicated.jsonl"

with open(gold_path, encoding="utf-8") as f:
    gold = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("HUMAN GOLD LABELS:")
for label, count in sorted(
    Counter(row["label"] for row in gold).items()
):
    print(f"{label}: {count}")

print("\nMODEL PREDICTIONS:")
for label, count in sorted(Counter(predictions).items()):
    print(f"{label}: {count}")


In [ ]:
import csv
import json
from sklearn.metrics import accuracy_score, classification_report

mapping = {
    "artistic passion": "fame",
    "generational inequality": "social",
    "joy": "celebration",
    "longing": "longing",
    "love": "romance",
    "loving oneself": "self_love",
    "mutual comfort": "friendship",
    "youth": "youth",
}

with open(ROOT / "gold_adjudicated.jsonl", encoding="utf-8") as f:
    gold = {
        row["item_id"]: row["label"]
        for line in f if line.strip()
        for row in [json.loads(line)]
    }

with open(ROOT / "mp9_faiss_predictions.csv", encoding="utf-8") as f:
    predictions_by_id = {
        row["item_id"]: row["predicted_label"]
        for row in csv.DictReader(f)
    }

ids = sorted(set(gold) & set(predictions_by_id))

assert len(ids) == 50, f"Expected 50 matching IDs, found {len(ids)}"

y_true = [mapping[gold[i]] for i in ids]
y_pred = [predictions_by_id[i] for i in ids]

print("Evaluated songs:", len(ids))
print("Mapped-label accuracy:", round(accuracy_score(y_true, y_pred), 4))
print()
print(classification_report(y_true, y_pred, zero_division=0))


In [ ]:
import json
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

# Existing MP7 predictions, matched to the human-labeled songs
baseline_df = pd.DataFrame({
    "item_id": ids,
    "human_label": [gold[i] for i in ids],
    "mapped_human_label": y_true,
    "mp7_prediction": y_pred
})

baseline_df["correct"] = (
    baseline_df["mapped_human_label"]
    == baseline_df["mp7_prediction"]
)

baseline_results = {
    "model": "MP7 fixed-exemplar KNN",
    "n_exemplars": len(exemplars),
    "n_gold": len(ids),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_f1": float(
        f1_score(y_true, y_pred, average="macro", zero_division=0)
    ),
    "taxonomy_mapping": mapping,
    "limitations": [
        "Human and MP7 label taxonomies differ",
        "Some semantic mappings are approximate",
        "Gold song descriptions may contain limited thematic information"
    ]
}

baseline_df.to_csv(
    ROOT / "mp9_baseline_predictions.csv",
    index=False
)

with open(ROOT / "mp9_baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)

print("Baseline accuracy:", baseline_results["accuracy"])
print("Baseline macro F1:", round(baseline_results["macro_f1"], 4))
print("Saved baseline predictions and results.")


In [ ]:
import numpy as np
import pandas as pd
import json
from sklearn.metrics import accuracy_score, f1_score

# Uses the FAISS index, vectors, and df from your retrieval experiment.
# Every song is classified using its 5 nearest OTHER songs.

k = 5
distances, neighbors = index.search(vectors, k + 1)

rag_predictions = []

for i in range(len(df)):
    nearest = [
        int(j) for j in neighbors[i]
        if j >= 0 and j != i
    ][:k]

    # Similarity-weighted vote using human labels
    votes = {}

    for j in nearest:
        label = df.iloc[j]["label"]
        similarity = float(np.dot(vectors[i], vectors[j]))
        weight = max(similarity, 0.0)

        votes[label] = votes.get(label, 0.0) + weight

    if votes and max(votes.values()) > 0:
        predicted = sorted(
            votes,
            key=lambda label: (-votes[label], label)
        )[0]
    else:
        predicted = df.iloc[nearest[0]]["label"]

    rag_predictions.append(predicted)

rag_df = df[["item_id", "label"]].copy()
rag_df["rag_prediction"] = rag_predictions

rag_accuracy = accuracy_score(
    rag_df["label"], rag_df["rag_prediction"]
)

rag_f1 = f1_score(
    rag_df["label"],
    rag_df["rag_prediction"],
    average="macro",
    zero_division=0
)

rag_df.to_csv(ROOT / "mp9_rag_predictions.csv", index=False)

print("Retrieval-augmented accuracy:", round(rag_accuracy, 4))
print("Retrieval-augmented macro F1:", round(rag_f1, 4))
print("MP7 baseline accuracy:", baseline_results["accuracy"])
print("MP7 baseline macro F1:", round(baseline_results["macro_f1"], 4))


In [ ]:
import numpy as np
import pandas as pd

# Match both experiments by song ID.
comparison = baseline_df[
    ["item_id", "correct"]
].merge(
    rag_df[["item_id", "label", "rag_prediction"]],
    on="item_id",
    validate="one_to_one"
)

baseline_correct = comparison["correct"].to_numpy(dtype=float)

rag_correct = (
    comparison["label"] == comparison["rag_prediction"]
).to_numpy(dtype=float)

# Paired bootstrap: resample the same songs for both models.
rng = np.random.default_rng(42)
n = len(comparison)

differences = []

for _ in range(5000):
    sample = rng.integers(0, n, size=n)

    difference = (
        rag_correct[sample].mean()
        - baseline_correct[sample].mean()
    )

    differences.append(difference)

lower, upper = np.percentile(differences, [2.5, 97.5])

print("Songs compared:", n)
print("Baseline accuracy:", round(baseline_correct.mean(), 4))
print("Retrieval accuracy:", round(rag_correct.mean(), 4))
print("Accuracy difference:", round(rag_correct.mean() - baseline_correct.mean(), 4))
print("95% bootstrap CI:", (round(lower, 4), round(upper, 4)))

pd.DataFrame({
    "accuracy_difference": differences
}).to_csv(ROOT / "mp9_comparison_bootstrap.csv", index=False)


In [ ]:
import json
import pandas as pd
from collections import Counter

# Load the 1,000 genuine text chunks
with open(ROOT / "mp9_unlabeled_chunks.jsonl", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

texts = [row["text"] for row in chunks]

assert len(chunks) == 1000
assert len(set(texts)) == 1000

# Reuse your already-trained MP7 classifier
silver_predictions = classifier.predict(texts)

print("Prediction count:", len(silver_predictions))
print("First prediction:", silver_predictions[0])


In [ ]:
import json
import re
from collections import Counter

LABEL_KEYWORDS = {
    "youth": ["teen", "school", "student", "young", "childhood"],
    "friendship": ["friend", "companionship", "together", "support"],
    "romance": ["romance", "romantic", "dating", "lover"],
    "self_love": ["self-esteem", "self-worth", "self-acceptance"],
    "longing": ["miss", "missing", "yearning", "nostalgia"],
    "fame": ["famous", "celebrity", "stardom", "superstar"],
    "social": ["inequality", "protest", "discrimination", "injustice"],
    "celebration": ["celebrate", "celebration", "festival", "party"]
}

def keyword_lf(text):
    text = text.lower()
    scores = {
        label: sum(
            bool(re.search(r"\b" + re.escape(word) + r"\b", text))
            for word in words
        )
        for label, words in LABEL_KEYWORDS.items()
    }

    best = max(scores.values())
    winners = [label for label, score in scores.items() if score == best]

    return winners[0] if best > 0 and len(winners) == 1 else None

def phrase_lf(text):
    text = text.lower()

    phrases = {
        "youth": ["growing up", "high school"],
        "friendship": ["close friends", "help each other"],
        "romance": ["fall in love", "love story"],
        "self_love": ["love yourself", "believe in yourself"],
        "longing": ["wish you were here", "miss you"],
        "fame": ["rise to fame", "music industry"],
        "social": ["social justice", "economic inequality"],
        "celebration": ["good news", "birthday party"]
    }

    matches = [
        label for label, patterns in phrases.items()
        if any(p in text for p in patterns)
    ]

    return matches[0] if len(matches) == 1 else None

assert len(chunks) == len(silver_predictions) == 1000

silver_rows = []

for chunk, model_label in zip(chunks, silver_predictions):
    lf1 = keyword_lf(chunk["text"])
    lf2 = phrase_lf(chunk["text"])

    votes = Counter()
    votes[str(model_label)] += 1.0

    if lf1:
        votes[lf1] += 1.0
    if lf2:
        votes[lf2] += 1.5

    ranking = votes.most_common()
    winner, top_score = ranking[0]
    second_score = ranking[1][1] if len(ranking) > 1 else 0

    # Abstain when evidence conflicts without a clear winner.
    silver_label = winner if top_score > second_score else None

    silver_rows.append({
        **chunk,
        "model_prediction": str(model_label),
        "keyword_lf": lf1,
        "phrase_lf": lf2,
        "silver_label": silver_label,
        "source_label": "silver" if silver_label else "abstain"
    })

output = ROOT / "mp9_silver_labels.jsonl"

with open(output, "w", encoding="utf-8") as f:
    for row in silver_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Total chunks:", len(silver_rows))
print("Silver labels:", sum(r["silver_label"] is not None for r in silver_rows))
print("Abstentions:", sum(r["silver_label"] is None for r in silver_rows))
print("Label distribution:")
print(Counter(r["silver_label"] for r in silver_rows))
print("Saved:", output)


In [ ]:
import json
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score


with open(ROOT / "gold_adjudicated.jsonl", encoding="utf-8") as f:
    gold_rows = [json.loads(line) for line in f if line.strip()]

gold_df = pd.DataFrame(gold_rows)
text_df = pd.read_csv(ROOT / "gold_text.csv")

eval_df = gold_df.merge(
    text_df[["item_id", "text"]],
    on="item_id",
    how="inner",
    validate="one_to_one"
)


label_mapping = {
    "artistic passion": "fame",
    "generational inequality": "social",
    "joy": "celebration",
    "longing": "longing",
    "love": "romance",
    "loving oneself": "self_love",
    "mutual comfort": "friendship",
    "youth": "youth"
}

eval_df["mapped_gold"] = eval_df["label"].map(label_mapping)

# Confirm that all human labels were mapped
assert eval_df["mapped_gold"].notna().all(), \
    "Some gold labels could not be mapped."

print("Human gold items:", len(eval_df))


texts = eval_df["text"].fillna("").astype(str).tolist()

model_preds = classifier.predict(texts)

assert len(model_preds) == len(eval_df)


def weak_vote(text, model_label):
    votes = {}

    def add_vote(label, weight):
        if label is not None:
            label = str(label)
            votes[label] = votes.get(label, 0.0) + weight

    # MP7 classifier prediction
    add_vote(model_label, 1.0)

    # Keyword-based labeling function
    add_vote(keyword_lf(text), 1.0)

    # Phrase-based labeling function
    add_vote(phrase_lf(text), 1.5)

    if not votes:
        return None

    ranked = sorted(
        votes.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Abstain if the highest scores are tied
    if len(ranked) > 1 and ranked[0][1] == ranked[1][1]:
        return None

    return ranked[0][0]

eval_df["model_prediction"] = [str(p) for p in model_preds]

eval_df["silver_prediction"] = [
    weak_vote(text, pred)
    for text, pred in zip(texts, model_preds)
]


valid = eval_df.dropna(
    subset=["mapped_gold", "silver_prediction"]
).copy()

total = len(eval_df)
evaluated = len(valid)
abstentions = total - evaluated

print("\nWEAK-SUPERVISION AGREEMENT RESULTS")
print("----------------------------------")
print("Gold items:", total)
print("Evaluated:", evaluated)
print("Abstentions:", abstentions)
print("Coverage:", round(evaluated / total, 4) if total else 0)

if evaluated > 0:

    agreement_accuracy = accuracy_score(
        valid["mapped_gold"],
        valid["silver_prediction"]
    )

    macro_f1 = f1_score(
        valid["mapped_gold"],
        valid["silver_prediction"],
        average="macro",
        zero_division=0
    )

    print("Agreement accuracy:", round(agreement_accuracy, 4))
    print("Macro F1:", round(macro_f1, 4))

else:
    print("No non-abstaining predictions to evaluate.")


output_path = ROOT / "mp9_silver_gold_agreement.csv"

eval_df.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)


display(
    eval_df[
        [
            "item_id",
            "label",
            "mapped_gold",
            "model_prediction",
            "silver_prediction"
        ]
    ].head(10)
)
